<a href="https://colab.research.google.com/github/jhlopesalves/eda_humanidades/blob/main/encontro_3/enc_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Google Colab"/></a>

<p class="lesson-runtime-note">No site, as saídas são resultados salvos. Para executar o código, abra o notebook no Colab.</p>


# Encontro 3: Funções e métricas lexicais

Neste encontro, reutilizamos as estruturas de Python introduzidas anteriormente
para construir procedimentos mais sistemáticos de processamento de textos.

Partimos do pré-processamento e da contagem lexical e avançamos até funções
reutilizáveis para calcular e organizar métricas sobre os dados.

## Retomada do Encontro 2 {.unnumbered .unlisted}

No encontro anterior, utilizamos alguns recursos que serão retomados aqui:

- operações com strings, incluindo `.replace()` e `.split()`;
- listas, indexação e slicing;
- estruturas condicionais com `if`, `elif` e `else`;
- iteração com loops `for`;
- dicionários, acesso por chave e `.get()`.

Esses recursos já permitem construir um procedimento simples de
pré-processamento e contagem de dados textuais.

## 1. Pré-processamento e normalização de texto

O texto abaixo será utilizado como **dado textual** nos exemplos deste encontro.

Antes da tokenização, aplicamos algumas operações simples de normalização:

1. conversão para letras minúsculas;
2. substituição de alguns sinais de pontuação por espaços;
3. substituição de quebras de linha por espaços.

Em seguida, `.split()` segmenta o resultado por espaços em branco.

A lista obtida depende dessas decisões. Alterar as regras de normalização ou de
segmentação pode alterar tanto os próprios tokens quanto as contagens
calculadas a partir deles.

### Robert K. Merton e as normas da ciência

Robert K. Merton foi um dos principais formuladores da sociologia da ciência:
seu interesse incluía as instituições, os valores e as relações sociais que
organizam a produção de conhecimento. No ensaio que vamos ler, a pergunta é
quais normas orientam a atividade científica e permitem que seus resultados
sejam examinados coletivamente. Quatro normas são frequentemente resumidas
pelo acrônimo **CUDOS**:

| Sigla | Norma e ideia central |
|---|---|
| C | **Communism**: o conhecimento científico integra um patrimônio compartilhado, disponível para a investigação coletiva. |
| U | **Universalism**: afirmações são avaliadas por critérios impessoais, independentemente da identidade ou posição social de quem as propõe. |
| D | **Disinterestedness**: instituições científicas devem conter interesses exclusivamente pessoais e favorecer uma investigação responsável perante a comunidade. |
| OS | **Organized Skepticism**: afirmações permanecem abertas a exame crítico sistemático. |

Essas normas oferecem uma perspectiva sobre o trabalho que fazemos como
pesquisadores. Compartilhar procedimentos e resultados permite a outros
examiná-los; explicitar decisões de análise permite discutir a evidência com
critérios que vão além da autoridade de quem apresenta uma conclusão. As
normas descrevem expectativas institucionais, e sua formulação também abre
espaço para investigar como a prática científica se aproxima ou se afasta delas.

Para quem trabalha com linguagem, estatística e humanidades, o texto interessa
em dois níveis: discute as condições sociais da pesquisa e oferece um material
linguístico sobre o qual podemos realizar operações transparentes de
normalização, tokenização e contagem. Ao alterar uma regra de processamento,
poderemos observar como ela altera a representação do texto e os resultados
que serão comunicados.

<details>
<summary>Fonte do trecho</summary>

O ensaio de Robert K. Merton foi publicado originalmente em 1942 e
posteriormente republicado como “The Normative Structure of Science”, em
[*The Sociology of Science: Theoretical and Empirical Investigations*](https://press.uchicago.edu/ucp/books/book/chicago/S/bo28451565.html)
(1973, pp. 267–278). A transcrição utilizada neste notebook acompanha a versão
reproduzida na página
[Mertonian norms](https://en.wikipedia.org/wiki/Mertonian_norms).

</details>

In [1]:
#| classes: [output-scroll]

mertonian_norms = """
The norms are expressed in the form of prescriptions, proscriptions, preferences, and permissions. They are legitimatized in terms of institutional values. These imperatives, transmitted by precept and example and reenforced by sanctions are in varying degrees internalized by the scientist, thus fashioning his scientific conscience or, if one prefers the latter-day phrase, his super-ego… [This scientific ethos] can be inferred from the moral consensus of scientists as expressed in use and wont, in countless writings on the scientific spirit and in moral indignation directed toward contraventions of the ethos.

An examination of the ethos of modern science is only a limited introduction to a larger problem: the comparative study of the institutional structure of science. Although detailed monographs assembling the needed comparative materials are few and scattered, they provide some basis for the provisional assumption that "science is afforded opportunity for development in a democratic order which is integrated with the ethos of science." This is not to say that the pursuit of science is confined to democracies.
"""

texto_normalizado = (
    mertonian_norms.lower()
    .replace(",", " ")
    .replace(".", " ")
    .replace("\n", " ")
    .replace("[", " ")
    .replace("]", " ")
)

tokens_mertonian = texto_normalizado.split()

print(tokens_mertonian[:100])
print(len(tokens_mertonian))

['the', 'norms', 'are', 'expressed', 'in', 'the', 'form', 'of', 'prescriptions', 'proscriptions', 'preferences', 'and', 'permissions', 'they', 'are', 'legitimatized', 'in', 'terms', 'of', 'institutional', 'values', 'these', 'imperatives', 'transmitted', 'by', 'precept', 'and', 'example', 'and', 'reenforced', 'by', 'sanctions', 'are', 'in', 'varying', 'degrees', 'internalized', 'by', 'the', 'scientist', 'thus', 'fashioning', 'his', 'scientific', 'conscience', 'or', 'if', 'one', 'prefers', 'the', 'latter-day', 'phrase', 'his', 'super-ego…', 'this', 'scientific', 'ethos', 'can', 'be', 'inferred', 'from', 'the', 'moral', 'consensus', 'of', 'scientists', 'as', 'expressed', 'in', 'use', 'and', 'wont', 'in', 'countless', 'writings', 'on', 'the', 'scientific', 'spirit', 'and', 'in', 'moral', 'indignation', 'directed', 'toward', 'contraventions', 'of', 'the', 'ethos', 'an', 'examination', 'of', 'the', 'ethos', 'of', 'modern', 'science', 'is', 'only', 'a']
169


A sequência de chamadas a `.replace()` funciona para os caracteres
explicitamente tratados, mas exige que cada substituição seja especificada
manualmente. À medida que o número de caracteres a tratar cresce, o procedimento se torna mais extenso e mais sujeito a omissões.

Uma alternativa é procurar uma operação da própria linguagem ou da biblioteca
padrão que permita expressar essa transformação de forma mais geral.

Pesquisar soluções, ler documentação e testar pequenas adaptações fazem parte
do trabalho cotidiano de programação. Não precisamos memorizar toda a
linguagem: precisamos formular o problema e avaliar as operações encontradas.
Aqui, a pergunta é como tratar vários sinais de pontuação de uma só vez.

### Consultando uma solução da comunidade

Uma busca por esse problema nos leva à discussão
[Best way to strip punctuation from a string](https://stackoverflow.com/questions/265960/best-way-to-strip-punctuation-from-a-string),
no Stack Overflow, um espaço público de perguntas e respostas sobre programação.
Votos e comentários ajudam a encontrar respostas que outras pessoas leram,
reutilizaram e discutiram. São indícios úteis para orientar a leitura; a
verificação continua sendo parte do trabalho.

A resposta mais votada apresenta `translate()`, mas conserva uma chamada de
Python 2. Podemos ler a ideia, verificar sua interface na
[documentação atual de `str.translate()`](https://docs.python.org/3.14/builtins/stdtypes.html#str.translate)
e adaptar a operação ao nosso objetivo:

| Exemplo histórico (Python 2) | Adaptação usada aqui (Python 3) |
|---|---|
| `s.translate(None, string.punctuation)` remove sinais | `s.translate(tabela_pontuacao)` usa uma tabela que troca sinais por espaços |

Consultamos `help()` abaixo e a documentação de
[`str.maketrans()`](https://docs.python.org/3.14/builtins/stdtypes.html#str.maketrans)
para construir essa tabela. Depois executamos a adaptação sobre o mesmo texto
e inspecionamos os tokens: a comparação mostra o efeito da regra escolhida.

In [2]:
#| classes: [output-scroll]

help(str.translate)

Help on method descriptor translate:

translate(self, table, /) unbound builtins.str method
    Replace each character in the string using the given translation table.

      table
        Translation table, which must be a mapping of Unicode ordinals
        to Unicode ordinals, strings, or None.

    The table must implement lookup/indexing via __getitem__, for
    instance a dictionary or list.  If this operation raises
    LookupError, the character is left untouched.  Characters mapped to
    None are deleted.



`string.punctuation` é uma string que reúne os caracteres ASCII de pontuação
disponibilizados pelo módulo `string`. A expressão usada em `str.maketrans()`
constrói uma tabela que associa cada um desses caracteres a um espaço.

Com dois argumentos, `str.maketrans()` recebe duas strings de mesmo
comprimento: cada caractere da primeira é associado ao caractere
correspondente da segunda. Aqui, a segunda string é composta apenas de
espaços, produzida por repetição com `*`.

Substituir a pontuação por espaços, em vez de simplesmente removê-la, evita que
trechos separados por um sinal sejam concatenados. Por exemplo, a remoção do
hífen em `latter-day` produziria `latterday`, enquanto sua substituição por um
espaço produz dois tokens.

In [3]:
#| classes: [output-scroll]

import string

tabela_pontuacao = str.maketrans(
    string.punctuation,
    " " * len(string.punctuation),
)

mertonian_normalizado = (
    mertonian_norms.lower()
    .translate(tabela_pontuacao)
)

tokens_mertonian = mertonian_normalizado.split()

print(tokens_mertonian[:100])
print(len(tokens_mertonian))

['the', 'norms', 'are', 'expressed', 'in', 'the', 'form', 'of', 'prescriptions', 'proscriptions', 'preferences', 'and', 'permissions', 'they', 'are', 'legitimatized', 'in', 'terms', 'of', 'institutional', 'values', 'these', 'imperatives', 'transmitted', 'by', 'precept', 'and', 'example', 'and', 'reenforced', 'by', 'sanctions', 'are', 'in', 'varying', 'degrees', 'internalized', 'by', 'the', 'scientist', 'thus', 'fashioning', 'his', 'scientific', 'conscience', 'or', 'if', 'one', 'prefers', 'the', 'latter', 'day', 'phrase', 'his', 'super', 'ego…', 'this', 'scientific', 'ethos', 'can', 'be', 'inferred', 'from', 'the', 'moral', 'consensus', 'of', 'scientists', 'as', 'expressed', 'in', 'use', 'and', 'wont', 'in', 'countless', 'writings', 'on', 'the', 'scientific', 'spirit', 'and', 'in', 'moral', 'indignation', 'directed', 'toward', 'contraventions', 'of', 'the', 'ethos', 'an', 'examination', 'of', 'the', 'ethos', 'of', 'modern', 'science', 'is']
170


::: {.callout-warning}
### Por que `ego…` ainda aparece assim?

`string.punctuation` reúne apenas caracteres de pontuação ASCII. Caracteres
Unicode como `…`, aspas curvas (`“ ”`, `‘ ’`) e travessões como `—` não
pertencem a essa string e, portanto, não são substituídos pela tabela construída
acima.

O tratamento da pontuação depende dos dados e do objetivo da análise. “Remover
pontuação” não constitui uma operação universal: diferentes regras de
normalização podem produzir diferentes tokens e, consequentemente, diferentes
contagens.
:::

## 2. Testando propriedades de strings

Alguns métodos de `str` permitem testar propriedades de uma string e retornam
um valor booleano (`True` ou `False`).

Neste encontro, utilizaremos quatro deles:

- `.startswith()` testa se a string começa com um prefixo especificado;
- `.isalpha()` testa se todos os caracteres são alfabéticos e se há pelo menos
  um caractere;
- `.isdigit()` testa se todos os caracteres são classificados como dígitos e se
  há pelo menos um caractere;
- `.isspace()` testa se todos os caracteres são classificados como espaço em
  branco e se há pelo menos um caractere.

Por essa razão, `.isalpha()`, `.isdigit()` e `.isspace()` retornam `False` para
uma string vazia.

Esses testes fornecem peças para construir um pipeline de limpeza e
organização de corpus: podemos reconhecer formas e decidir quais seguirão
para uma etapa de processamento. Seus resultados podem ser combinados com
`if`, `and`, `or` e `not`.

| Método | Teste | Uso no corpus |
|---|---|---|
| `.startswith(prefixo)` | começa com o prefixo indicado? | detectar URLs, hashtags ou marcadores |
| `.isalpha()` | tem apenas caracteres alfabéticos e não é vazia? | selecionar formas feitas só de letras |
| `.isdigit()` | tem apenas dígitos e não é vazia? | detectar tokens numéricos |
| `.isspace()` | tem apenas espaços em branco e não é vazia? | localizar espaços ou trechos vazios |

A regra útil depende da análise. Uma hashtag, por exemplo, pode ser justamente
a unidade que queremos conservar. Vamos consultar cada método e experimentar
pequenos casos para conhecer seu comportamento.

In [4]:
#| classes: [output-scroll]

help(str.startswith)

Help on method descriptor startswith:

startswith(self, prefix[, start[, end]], /) unbound builtins.str method
    Return True if the string starts with the specified prefix, False otherwise.

    prefix
      A string or a tuple of strings to try.
    start
      Optional start position. Default: start of the string.
    end
      Optional stop position. Default: end of the string.



In [5]:
print("Python".startswith("Py"))
print("#hashtag".startswith("#"))
print("https://exemplo.com".startswith("http"))
print("linguagem".startswith("#"))

True
True
True
False


In [6]:
#| classes: [output-scroll]

help(str.isalpha)

Help on method descriptor isalpha:

isalpha(self, /) unbound builtins.str method
    Return True if the string is an alphabetic string, False otherwise.

    A string is alphabetic if all characters in the string are
    alphabetic and there is at least one character in the string.



In [7]:
print("linguagem".isalpha())
print("abc123".isalpha())
print("não".isalpha())

True
False
True


O resultado para `"não"` é `True`: `.isalpha()` não se limita às letras ASCII
sem acentos. Caracteres alfabéticos Unicode, como `ã`, também são reconhecidos
como alfabéticos. Isso torna o teste útil em corpora multilíngues, nos quais
as formas lexicais não se restringem ao alfabeto ASCII. O resultado descreve
os caracteres da string; a regra de seleção do corpus continua sendo uma
decisão da análise.

In [8]:
#| classes: [output-scroll]

help(str.isdigit)

Help on method descriptor isdigit:

isdigit(self, /) unbound builtins.str method
    Return True if the string is a digit string, False otherwise.

    A string is a digit string if all characters in the string are
    digits and there is at least one character in the string.



In [9]:
print("1921".isdigit())
print("3a".isdigit())
print("".isdigit())

True
False
False


In [10]:
#| classes: [output-scroll]

help(str.isspace)

Help on method descriptor isspace:

isspace(self, /) unbound builtins.str method
    Return True if the string is a whitespace string, False otherwise.

    A string is whitespace if all characters in the string are
    whitespace and there is at least one character in the string.



In [11]:
print("1               ".isspace())
print("    a".isspace())
print(" ".isspace())
print("\t\n".isspace())

False
False
True
True


## 3. Contagens de frequência com dicionários

Após a tokenização, `tokens_mertonian` contém uma sequência de ocorrências:
cada elemento corresponde a um **token** observado no texto.

Para descrever a frequência dessas formas, podemos construir um dicionário no
qual:

- cada chave representa um **tipo lexical** (*type*), isto é, uma forma distinta
  observada entre os tokens;
- cada valor representa sua **frequência absoluta**, isto é, o número de
  ocorrências dessa forma no texto.

Aqui, **tipo lexical** é um conceito linguístico e não deve ser confundido com
o tipo computacional de um objeto em Python, consultado, por exemplo, com
`type()`.

Para construir o dicionário, precisamos também de uma operação ainda não
utilizada diretamente nos exemplos anteriores. A atribuição:

```python
dicionario[chave] = valor
```
associa um valor a uma chave. Se a chave ainda não existir, uma nova entrada é
criada; se ela já existir, o valor associado à chave é substituído.

Durante uma contagem, podemos utilizar essa operação para registrar a primeira
ocorrência de um token e atualizar sua frequência nas ocorrências seguintes.
Primeiro faremos isso explicitamente com `if` e `else`; depois escreveremos o
mesmo procedimento de forma mais compacta com `.get()`.

Considere a sequência curta `["the", "norms", "the"]`. O dicionário começa
vazio e acumula uma contagem a cada ocorrência:

| Token | Antes | Ação | Depois |
|---|---|---|---|
| `the` | ausente | criar chave | `1` |
| `norms` | ausente | criar chave | `1` |
| `the` | `1` | somar `1` | `2` |

A primeira ocorrência cria uma entrada. A repetição atualiza a mesma entrada,
sem criar outro tipo lexical. Ao final, temos `{"the": 2, "norms": 1}`: três
tokens, dois tipos. O loop seguinte realiza essas decisões para o texto inteiro.

In [12]:
frequencias_explicitas = {}

for token in tokens_mertonian:
    if token in frequencias_explicitas:
        frequencias_explicitas[token] += 1
    else:
        frequencias_explicitas[token] = 1

print(frequencias_explicitas["the"])
print(frequencias_explicitas["science"])

14
5


### Atualização de frequências com `.get()`

O mesmo procedimento pode ser expresso de forma mais compacta com `.get()`.

Durante a contagem, a frequência atual de um tipo lexical pode estar em uma de
duas situações:

- se a chave já existe, seu valor corresponde ao número de ocorrências contadas
  até aquele ponto;
- se a chave ainda não existe, nenhuma ocorrência foi registrada antes da
  ocorrência atual e podemos tomar sua frequência acumulada como `0`.

Assim, a expressão:

```python
frequencias.get(token, 0) + 1
```

recupera a frequência atual - ou 0 para uma chave ainda ausente - e acrescenta
a nova ocorrência.

Podemos então associar esse novo valor à própria chave:
```python
frequencias[token] = frequencias.get(token, 0) + 1
```

In [13]:
frequencias = {}

for token in tokens_mertonian:
    frequencias[token] = frequencias.get(token, 0) + 1

print(frequencias == frequencias_explicitas)

True


### Iteração sobre tipos e frequências com `.items()`

O método `.items()` permite percorrer simultaneamente as chaves e os valores de
um dicionário.

No dicionário `frequencias`, cada par corresponde a:

`(tipo, frequência)`

Ao escrever:

`for tipo, frequencia in frequencias.items():`

Python separa cada par em duas variáveis. Esse procedimento é chamado de
**unpacking**.

A partir desses dois valores, podemos combinar propriedades do tipo lexical com
sua frequência observada.

In [14]:
for tipo, frequencia in frequencias.items():
    if len(tipo) > 8 and frequencia > 1:
        print(f"{tipo}: {frequencia}")

expressed: 2
institutional: 2
scientific: 3
comparative: 2


Os critérios utilizados acima - mais de oito caracteres e frequência superior
a uma ocorrência - servem apenas para demonstrar a combinação entre
informações do dicionário e uma condição lógica. Eles não definem uma categoria
linguística ou estatística particular.

Python aceita identificadores Unicode, inclusive letras acentuadas. Neste
exemplo usamos `frequencia`; mais adiante, `calcular_média`. Em um mesmo
programa, mantenha a grafia consistente em todas as referências a cada nome.

## 4. Primeiras medidas descritivas

Os objetos construídos até aqui - tokens, frequências e comprimentos - podem
ser resumidos por medidas numéricas.

Com listas, dicionários, loops e operações aritméticas, já podemos calcular
medidas lexicais e estatísticas que aparecem em pesquisas. Vamos definir cada
quantidade, tornar seu cálculo visível e interpretar o que ela resume. Essa
transparência permite também reconhecer o que uma medida deixa de representar.

### 4.1 Type-token ratio (TTR)

A **type-token ratio** (TTR) relaciona o número de formas distintas observadas
ao número total de tokens:

$$
\text{TTR}
=
\frac{\text{número de tipos}}{\text{número de tokens}}
$$

Em `tokens_mertonian`, cada elemento da lista corresponde a uma ocorrência
(token), enquanto cada chave de `frequencias` corresponde a uma forma distinta
(type).

Para uma sequência finita não vazia com $N$ tokens, há pelo menos um e no máximo $N$ tipos lexicais. Portanto:

$$\frac{1}{N} \leq \mathrm{TTR} \leq 1.$$

O limite inferior corresponde a repetir uma única forma em todos os $N$
tokens; o superior corresponde a ter uma forma distinta em cada token. Para
uma sequência vazia, a razão não está definida. Uma TTR de `1` indica que todas
as formas são distintas; a repetição reduz a proporção.

Por exemplo, `["data", "science", "data", "language"]` contém quatro tokens e
três tipos: `data`, `science` e `language`. Sua TTR é `3 / 4 = 0.75`. Podemos
alongar esse pequeno texto repetindo a mesma sequência:

| Sequência | Tokens / tipos | TTR |
|---|---|---|
| sequência original | `4` / `3` | `0.75` |
| sequência repetida duas vezes | `8` / `3` | `0.375` |
| sequência repetida quatro vezes | `16` / `3` | `0.1875` |

::: {.callout-note}
### Comprimento do texto e TTR

A medida depende das regras de pré-processamento e do número de tokens. No
exemplo, o vocabulário permanece igual e a proporção diminui apenas porque há
mais repetições. Em textos maiores, formas já observadas tendem a reaparecer.
Por isso, comparar TTR bruta entre textos de tamanhos muito diferentes mistura
diferenças de vocabulário com diferenças de extensão; comprimentos semelhantes
também não eliminam completamente essa dependência.

MATTR e MTLD são alternativas que procuram reduzir a influência do comprimento.
Seus cálculos ficam para outro momento.
:::

A próxima célula usa as contagens do trecho de Merton: o denominador é o número
de ocorrências e o numerador é o número de chaves do dicionário de frequências.

In [15]:
numero_tokens = len(tokens_mertonian)
numero_tipos = len(frequencias)

valor_ttr = numero_tipos / numero_tokens

print(f"Tokens: {numero_tokens}")
print(f"Tipos: {numero_tipos}")
print(f"TTR: {valor_ttr:.3f}")

Tokens: 170
Tipos: 106
TTR: 0.624


### 4.2 Amplitude

A **amplitude** (*range*) descreve a extensão entre o menor e o maior valor
observado de uma variável quantitativa.

Neste exemplo, cada observação corresponde a uma ocorrência de token em
`tokens_mertonian`, e a variável analisada é o comprimento desse token em
caracteres.

Para $n > 0$ valores $x_1, x_2, \ldots, x_n$, a amplitude é definida por:

$$
\text{amplitude}
=
\max(x) - \min(x)
$$

A amplitude utiliza apenas os dois valores extremos da distribuição. Por isso,
uma única observação muito pequena ou muito grande pode alterar
substancialmente seu valor, mesmo que as demais observações permaneçam
inalteradas.

Ela informa a extensão total dos valores observados, mas não descreve como as
observações se distribuem entre os extremos.

In [16]:
comprimentos = []

for token in tokens_mertonian:
    comprimentos.append(len(token))

print(comprimentos[:20])
print(f"Número de observações: {len(comprimentos)}")

[3, 5, 3, 9, 2, 3, 4, 2, 13, 13, 11, 3, 11, 4, 3, 13, 2, 5, 2, 13]
Número de observações: 170


A lista `comprimentos` preserva uma observação para cada token, mas substitui a
forma textual por uma medida numérica: seu comprimento.

A partir dessa representação, `min()` e `max()` identificam os extremos
observados, e sua diferença fornece a amplitude.

In [17]:
comprimento_minimo = min(comprimentos)
comprimento_maximo = max(comprimentos)

amplitude = comprimento_maximo - comprimento_minimo

print(f"Comprimento mínimo: {comprimento_minimo}")
print(f"Comprimento máximo: {comprimento_maximo}")
print(f"Amplitude: {amplitude}")

Comprimento mínimo: 1
Comprimento máximo: 14
Amplitude: 13


Neste conjunto de tokens, os comprimentos observados variam de 1 a 14
caracteres, resultando em uma amplitude de 13 caracteres.

Esse valor descreve apenas a distância entre os extremos. Distribuições muito
diferentes podem apresentar a mesma amplitude se compartilharem o mesmo mínimo
e o mesmo máximo:

| Valores | Mín. | Máx. | Amplitude |
|---|---|---|---|
| `[1, 2, 3, 4, 5]` | `1` | `5` | `4` |
| `[1, 1, 1, 1, 5]` | `1` | `5` | `4` |

O segundo conjunto concentra quatro observações no mínimo. A amplitude não
registra essa concentração: os dois extremos determinam o resultado.

### 4.3 Mediana

A **mediana** é uma medida de posição determinada a partir dos valores
ordenados.

Para uma sequência numérica não vazia, seu cálculo é:

1. ordenamos as observações;
2. se o número de observações $n$ for ímpar, selecionamos o valor da posição
   central;
3. se $n$ for par, calculamos a média dos dois valores centrais.

Veja os casos que serão executados nas células seguintes, já ordenados:

| Ímpar: `n = 5` | Par: `n = 4` |
|---|---|
| `[2, 3, 5, 7, 9]` | `[2, 3, 7, 9]` |
| posição central: `3` | posições centrais: `2` e `3` |
| índice Python: `2` | índices Python: `1` e `2` |
| mediana: `5` | mediana: `(3 + 7) / 2 = 5` |

Depois da ordenação, `n // 2` fornece o índice usado pela implementação. Para
`n = 5`, esse índice é `2`, que seleciona o terceiro valor. Para `n = 4`, é
`2`, que seleciona o valor central à direita; o outro está em `2 - 1 = 1`.
A divisão inteira `//` e a indexação iniciada em zero explicam os dois casos.

A mediana é uma **estatística resistente**: sua posição depende da ordenação dos
dados, e não diretamente da magnitude de todos os valores observados. Alterar
um valor já localizado em uma das extremidades da distribuição pode modificar
substancialmente sua magnitude sem alterar a mediana.

A média e a mediana resumem a localização da distribuição de maneiras
diferentes. A média utiliza os valores de todas as observações e, por isso, é
mais sensível a valores extremos; a mediana é determinada pelas observações
centrais depois da ordenação.

In [18]:
# Caso par: n = 4
valores = [3, 7, 2, 9]

valores_ordenados = sorted(valores)
indice_do_meio = len(valores_ordenados) // 2

meio_esquerda = valores_ordenados[indice_do_meio - 1]
meio_direita = valores_ordenados[indice_do_meio]

mediana = (meio_esquerda + meio_direita) / 2

print(valores_ordenados)
print(mediana)

[2, 3, 7, 9]
5.0


In [19]:
# Caso ímpar: n = 5
valores = [3, 7, 2, 9, 5]

valores_ordenados = sorted(valores)
indice_do_meio = len(valores_ordenados) // 2

mediana = valores_ordenados[indice_do_meio]

print(valores_ordenados)
print(mediana)

[2, 3, 5, 7, 9]
5


## 5. Funções e código reutilizável

Uma **função** reúne uma sequência de operações sob um nome. Depois de
definida, ela pode ser executada repetidamente com diferentes valores de
entrada.

As operações de média, mediana e TTR que já compreendemos podem receber nomes
e ser reaplicadas de maneira consistente. Uma definição simples utiliza `def`:

| Elemento | Papel |
|---|---|
| `def` | inicia a definição de uma função |
| nome | identifica o procedimento que poderá ser chamado |
| parâmetro | nome que recebe um valor fornecido na chamada |
| argumento | valor concreto fornecido ao chamar a função |
| bloco indentado | operações executadas quando a função é chamada |
| `return` | devolve o resultado ao código que chamou a função |

Na definição `def calcular_média(valores):`, `valores` é um parâmetro. Em uma
chamada como `calcular_média([2, 4, 6])`, a lista fornecida é o argumento
associado a esse parâmetro.

A chamada de uma função executa seu corpo e, quando há uma instrução `return`,
produz o valor devolvido pela função.

As funções desta seção operam sobre entradas não vazias e pressupõem argumentos
compatíveis com o cálculo realizado.

### 5.1 Uma função para a média

Para $n > 0$ valores $x_1, x_2, \ldots, x_n$, a média aritmética é:

$$
\bar{x}
=
\frac{1}{n}\sum_{i=1}^{n}x_i
$$

A implementação requer apenas a soma dos valores e o número de observações.
Encapsular esse cálculo em uma função permite aplicá-lo a diferentes conjuntos
de valores sem repetir as mesmas instruções.

In [20]:
def calcular_média(valores: list[float]) -> float:
    return sum(valores) / len(valores)

Uma definição de função pode indicar também os tipos esperados para seus
parâmetros e para o valor retornado. Em:

`def calcular_média(valores: list[float]) -> float:`

`valores: list[float]` indica que o parâmetro recebe uma lista de números em
ponto flutuante, enquanto `-> float` indica o tipo esperado para o valor
retornado.

Essas anotações tornam a interface da função mais explícita e facilitam a
leitura, a documentação e a análise estática do código.

Python não verifica essas anotações durante a execução: uma lista de inteiros,
por exemplo, é aceita normalmente. As anotações documentam a intenção do autor
e são exploradas por ferramentas de análise estática, não pelo interpretador.

### 5.2 Uma função para a mediana

Considere os valores ordenados:

$$
x_1 \leq x_2 \leq \cdots \leq x_n
$$

A mediana depende de o número de observações ser ímpar ou par:

$$
\text{mediana}
=
\begin{cases}
x_{\frac{n+1}{2}},
& \text{se } n \text{ é ímpar} \\[6pt]
\frac{x_{\frac{n}{2}} + x_{\frac{n}{2}+1}}{2},
& \text{se } n \text{ é par}
\end{cases}
$$

A implementação segue os mesmos dois casos examinados anteriormente. Há apenas
uma diferença de representação: na notação matemática, as posições começam em
1; em Python, os índices começam em 0.

In [21]:
def calcular_mediana(valores: list[float]) -> float:
    valores_ordenados = sorted(valores)
    n = len(valores_ordenados)
    indice_do_meio = n // 2

    if n % 2 != 0: # Caso ímpar
        return valores_ordenados[indice_do_meio]
    else: # Caso par
        meio_esquerda = valores_ordenados[indice_do_meio - 1]
        meio_direita = valores_ordenados[indice_do_meio]
        return (meio_esquerda + meio_direita) / 2

### 5.3 Uma função para a TTR

A TTR é definida por:

$$
\text{TTR}
=
\frac{\text{número de tipos}}{\text{número de tokens}}
$$

Para que a função dependa apenas da lista de tokens recebida, precisamos obter
as formas distintas diretamente dessa lista.

A função `set()` constrói um conjunto sem elementos repetidos. Assim,
`set(tokens)` contém uma ocorrência de cada forma distinta e
`len(set(tokens))` fornece o número de tipos.

O conjunto mantém cada valor distinto uma única vez. Compare:

| Lista de tokens | Formas distintas em um conjunto |
|---|---|
| `["data", "science", "data", "language"]` | `{"data", "science", "language"}` |

A lista registra a sequência de quatro ocorrências; o conjunto reúne três
formas distintas. A ordem não é o objetivo dessa representação, e sua
exibição pode variar. Aqui, `len(set(tokens))` conta o vocabulário observado,
enquanto `len(tokens)` conta todas as ocorrências.

In [22]:
def calcular_ttr(tokens: list[str]) -> float:
    numero_tokens = len(tokens)
    numero_tipos = len(set(tokens))

    return numero_tipos / numero_tokens

### Verificação com exemplos simples

Pequenos conjuntos de valores permitem verificar se cada função reproduz
resultados que podem ser determinados diretamente.

Para a mediana, convém verificar separadamente os casos com número ímpar e par
de observações.

In [23]:
média_exemplo = calcular_média(
    valores=[2, 4, 6],
)

mediana_impar = calcular_mediana(
    valores=[1, 2, 3, 4, 5],
)

mediana_par = calcular_mediana(
    valores=[1, 2, 3, 4],
)

ttr_exemplo = calcular_ttr(
    tokens=["a", "b", "a", "c"],
)

print(f"Média: {média_exemplo}")
print(f"Mediana, n ímpar: {mediana_impar}")
print(f"Mediana, n par: {mediana_par}")
print(f"TTR: {ttr_exemplo}")

Média: 4.0
Mediana, n ímpar: 3
Mediana, n par: 2.5
TTR: 0.75


In [24]:
média_comprimentos = calcular_média(
    valores=comprimentos,
)

mediana_comprimentos = calcular_mediana(
    valores=comprimentos,
)

ttr_mertonian = calcular_ttr(
    tokens=tokens_mertonian,
)

print(f"Média dos comprimentos: {média_comprimentos:.2f}")
print(f"Mediana dos comprimentos: {mediana_comprimentos}")
print(f"TTR: {ttr_mertonian:.3f}")

Média dos comprimentos: 5.53
Mediana dos comprimentos: 4.0
TTR: 0.624


In [25]:
token_mais_longo = max(tokens_mertonian, key=len)

print(f"Token mais longo: {token_mais_longo!r}")
print(f"Comprimento: {len(token_mais_longo)} caracteres")

Token mais longo: 'contraventions'
Comprimento: 14 caracteres


Em `max(tokens_mertonian, key=len)`, o argumento `key` recebe a própria função
`len` - sem parênteses, pois não a estamos chamando. `max()` aplica essa função
a cada elemento e compara os resultados, mas retorna o elemento original: o
token mais longo, não seu comprimento.

## Síntese

Construímos um percurso de normalização, tokenização e contagem de
frequências, primeiro com operações explícitas e depois com recursos mais
gerais da linguagem. Dicionários distinguiram ocorrências e formas
lexicais; funções permitiram reutilizar os cálculos de média, mediana e TTR.

Os resultados dependem das regras de processamento e do domínio de cada
medida. O comprimento dos tokens é uma variável derivada; a TTR também
depende fortemente da extensão do texto. Calcular uma medida corretamente
não basta para justificar uma comparação.

**Atividade:** [Tarefa](https://colab.research.google.com/github/jhlopesalves/eda_humanidades/blob/main/encontro_3/homework/enc_3_homework.ipynb) · [Gabarito](https://colab.research.google.com/github/jhlopesalves/eda_humanidades/blob/main/encontro_3/homework/enc_3_respostas_hw.ipynb) (Google Colab).
